In [1]:
# Output:Variable,
#     *,                                   !- Key (all surfaces)
#     Surface Inside Face Conduction Heat Transfer Energy,  !- Variable Name
#     Timestep;                               !- Reporting Frequency

# Output:Variable,
#     *,
#     Surface Window Heat Loss Energy,
#     Timestep;

# Output:Variable,
#     *,
#     Surface Window Heat Gain Energy,
#     Timestep;

# Output:Variable,
#     *,
#     Surface Window Net Heat Transfer Energy,
#     Timestep;

# Output:Variable,
#     *,
#     Surface Window Transmitted Solar Radiation Energy,
#     Timestep;

# Output:Variable,
#     *,
#     AFN Zone Infiltration Sensible Heat Loss Energy,
#     Timestep;

# Output:Variable,
#     *,
#     AFN Zone Ventilation Sensible Heat Loss Energy,
#     Timestep;

# Output:Variable,
#     *,
#     Zone Ideal Loads Zone Sensible Heating Energy,
#     Timestep;



#### Testing the build function

In [2]:
import pandas as pd
from cubes.construct.buildingconfig import load_building_config
from cubes.construct.building import Building
from cubes.package.envconfig import EnvConfig

files_dir = "/workspaces/CUBES/beizaee_validation/misc/input"
bc_file_path = "/workspaces/CUBES/cubes/data/buildingconfigs/thermostat_experiment/2023/case0.json"

materials = pd.read_pickle("/workspaces/CUBES/cubes/materials.pickle")
windows = pd.read_pickle("/workspaces/CUBES/cubes/windows.pickle")

BC = load_building_config(bc_file_path, files_dir=files_dir)
EC = EnvConfig(files_dir=files_dir)

factor_vertical = 1
factor_ext = 1

# specify which zones should have which type strengthened
internal_targets = {
    "subfloor": [
        "internal_floor",
        "internal_ceiling",
        "internal_wall",
        "internal_door"
    ],   # subfloor ceiling → ground floor
    "front_room": [
        "internal_floor",
        "internal_ceiling",
        "internal_wall",
        "internal_door"
        ],   # ground-floor rooms floor → subfloor
    "backroom": [
        "internal_floor",
        "internal_ceiling",
        "internal_wall",
        "internal_door"
    ],
    "kitchen": [
        "internal_floor",
        "internal_ceiling",
        "internal_wall",
        "internal_door"
        ],
    "hall_downstairs": [
        "internal_floor",
        "internal_ceiling",
        "internal_wall",
        "internal_door"
        ],
    "hall_upstairs": [
        "internal_floor",
        "internal_ceiling",
        "internal_wall",
        "internal_door"
        ],  # first floor ceiling → loft
    "bathroom": [
        "internal_floor",
        "internal_ceiling",
        "internal_wall",
        "internal_door"
        ],
    "bedroom_2": [
        "internal_floor",
        "internal_ceiling",
        "internal_wall",
        "internal_door"
        ],
    "loft": [
        "internal_floor",
        "internal_ceiling",
        "internal_wall",
        "internal_door"
        ]          # loft floor → first floor
}

# specify which zones should have which type strengthened
external_targets = {
    "subfloor": [
        # "internal_floor",
        # "internal_ceiling",
        # "internal_wall",
        # "internal_door"
    ],   # subfloor ceiling → ground floor
    "front_room": [
        "external_wall",
        "external_window",
        ],   # ground-floor rooms floor → subfloor
    "backroom": [
        "external_wall",
        "external_window",
    ],
    "kitchen": [
        "external_wall",
        "external_window",
        ],
    "hall_downstairs": [
        "external_wall",
        "external_window",
        ],
    "hall_upstairs": [
        "external_wall",
        "external_window",
        ],
    "bathroom": [
        "external_wall",
        "external_window",
        ],
    "bedroom_2": [
        "external_wall",
        "external_window",
        ],
    "loft": [
        "external_wall",
        "external_window",
        ]
}

for zone, cracks in BC.cracks.items():
    if isinstance(cracks, str) or zone.lower() == "default":
        continue
    z = zone.lower()
    if z not in internal_targets and z not in external_targets:
        continue
    for surf_type, entry in cracks.items():
        if not isinstance(entry, dict):
            continue
        if surf_type in internal_targets[z]:
            factor = factor_vertical
        elif surf_type in external_targets[z]:
            factor = factor_ext
        else:
            continue
        if "cq_per_m2_factor" in entry:
            entry["cq_per_m2_factor"] *= factor
        if "cq_per_m_factor" in entry:
            entry["cq_per_m_factor"] *= factor
        print(f"{zone}: {surf_type} ×{factor}")

building = Building(BC, materials, windows)
building.build()
idf = building.get_idf()
idf.save("test.idf")

subfloor: internal_ceiling ×1
front_room: external_wall ×1
front_room: internal_wall ×1
front_room: internal_floor ×1
front_room: internal_ceiling ×1
front_room: internal_door ×1
front_room: external_window ×1
backroom: external_wall ×1
backroom: internal_wall ×1
backroom: internal_floor ×1
backroom: internal_ceiling ×1
backroom: internal_door ×1
backroom: external_window ×1
kitchen: external_wall ×1
kitchen: internal_wall ×1
kitchen: internal_floor ×1
kitchen: internal_ceiling ×1
kitchen: internal_door ×1
kitchen: external_window ×1
hall_downstairs: external_wall ×1
hall_downstairs: internal_wall ×1
hall_downstairs: internal_floor ×1
hall_downstairs: internal_ceiling ×1
hall_downstairs: internal_door ×1
hall_downstairs: external_window ×1
bedroom_2: external_wall ×1
bedroom_2: internal_wall ×1
bedroom_2: internal_floor ×1
bedroom_2: internal_ceiling ×1
bedroom_2: internal_door ×1
bedroom_2: external_window ×1
bathroom: external_wall ×1
bathroom: internal_wall ×1
bathroom: internal_flo

/workspaces/CUBES/cubes/construct/utilities.py:68: RuntimeWarning: invalid value encountered in divide
  return n / np.linalg.norm(n)


🪟 WINDOW | Block front_room Storey 0 Wall 0004: placed (1.75×2.10 m, 3.66 m²) vs requested 3.66 m² | wall_area=10.08 m²
✅ Added window to front_room on Block front_room Storey 0 Wall 0004 (az=180°, 3.66 m²)
🪟 WINDOW | Block backroom Storey 0 Wall 0002: placed (1.38×1.66 m, 2.30 m²) vs requested 2.30 m² | wall_area=7.56 m²
✅ Added window to backroom on Block backroom Storey 0 Wall 0002 (az=0°, 2.30 m²)
🪟 WINDOW | Block kitchen Storey 0 Wall 0002: placed (0.87×1.04 m, 0.90 m²) vs requested 0.90 m² | wall_area=7.56 m²
✅ Added window to kitchen on Block kitchen Storey 0 Wall 0002 (az=0°, 0.90 m²)
🪟 WINDOW | Block bedroom_3 Storey 0 Wall 0004: placed (0.66×0.80 m, 0.53 m²) vs requested 0.53 m² | wall_area=6.86 m²
✅ Added window to bedroom_3 on Block bedroom_3 Storey 0 Wall 0004 (az=180°, 0.53 m²)
🪟 WINDOW | Block bedroom_1 Storey 0 Wall 0004: placed (1.56×1.87 m, 2.92 m²) vs requested 2.92 m² | wall_area=8.26 m²
✅ Added window to bedroom_1 on Block bedroom_1 Storey 0 Wall 0004 (az=180°, 2.9

In [3]:
# idf.rotate(90)
# idf.view_model()

In [4]:
# transplant_afn_geometry.py
from __future__ import annotations
from collections import defaultdict
from pathlib import Path
from typing import Iterable, Optional
from eppy.modeleditor import IDF

# ----------------- helpers -----------------
def _round_if_num(val, ndp: Optional[int]):
    try:
        f = float(val)
        return round(f, ndp) if ndp is not None else f
    except Exception:
        return val

def clone_building_surface(dst_idf: IDF, s, round_vertex_decimals: int = 6):
    """Clone BuildingSurface:Detailed with correct Number_of_Vertices & vertices."""
    payload = {}
    # copy non-vertex fields (skip key & Number_of_Vertices; we will set it)
    for fn in s.fieldnames:
        fl = fn.lower()
        if fl == "key" or fn == "Number_of_Vertices" or fl.startswith("vertex_"):
            continue
        val = getattr(s, fn)
        if val not in ("", None):
            payload[fn] = val
    # collect vertices
    verts = []
    for i in range(1, 121):
        x = getattr(s, f"Vertex_{i}_Xcoordinate", "")
        y = getattr(s, f"Vertex_{i}_Ycoordinate", "")
        z = getattr(s, f"Vertex_{i}_Zcoordinate", "")
        if x == "" and y == "" and z == "":
            break
        if x == "" or y == "" or z == "":
            break
        verts.append((
            _round_if_num(x, round_vertex_decimals),
            _round_if_num(y, round_vertex_decimals),
            _round_if_num(z, round_vertex_decimals),
        ))
    if len(verts) < 3:
        raise ValueError(f"[BuildingSurface] {s.Name}: has <3 vertices; check source.")
    payload["Number_of_Vertices"] = len(verts)
    for i, (x, y, z) in enumerate(verts, start=1):
        payload[f"Vertex_{i}_Xcoordinate"] = x
        payload[f"Vertex_{i}_Ycoordinate"] = y
        payload[f"Vertex_{i}_Zcoordinate"] = z
    return dst_idf.newidfobject(s.key, **payload)

def clone_fenestration_surface(dst_idf: IDF, f, round_vertex_decimals: int = 6):
    """Clone FenestrationSurface:Detailed with correct vertices."""
    payload = {}
    for fn in f.fieldnames:
        fl = fn.lower()
        if fl == "key" or fn == "Number_of_Vertices" or fl.startswith("vertex_"):
            continue
        val = getattr(f, fn)
        if val not in ("", None):
            payload[fn] = val
    verts = []
    for i in range(1, 121):
        x = getattr(f, f"Vertex_{i}_Xcoordinate", "")
        y = getattr(f, f"Vertex_{i}_Ycoordinate", "")
        z = getattr(f, f"Vertex_{i}_Zcoordinate", "")
        if x == "" and y == "" and z == "":
            break
        if x == "" or y == "" or z == "":
            break
        verts.append((
            _round_if_num(x, round_vertex_decimals),
            _round_if_num(y, round_vertex_decimals),
            _round_if_num(z, round_vertex_decimals),
        ))
    if len(verts) < 3:
        raise ValueError(f"[FenestrationSurface] {f.Name}: has <3 vertices; check source.")
    payload["Number_of_Vertices"] = len(verts)
    for i, (x, y, z) in enumerate(verts, start=1):
        payload[f"Vertex_{i}_Xcoordinate"] = x
        payload[f"Vertex_{i}_Ycoordinate"] = y
        payload[f"Vertex_{i}_Zcoordinate"] = z
    return dst_idf.newidfobject(f.key, **payload)

def _clone_obj(dst_idf: IDF, obj):
    """Clone a non-geometry EpBunch (no vertex management)."""
    payload = {
        fn: getattr(obj, fn)
        for fn in obj.fieldnames
        if fn.lower() != "key" and getattr(obj, fn) not in ("", None)
    }
    return dst_idf.newidfobject(obj.key, **payload)

def _names(objs) -> set[str]:
    return {o.Name for o in objs} if objs else set()

def _ensure_basics(dst: IDF):
    """Ensure ReferenceCrackConditions and AlwaysOn/Off schedules exist."""
    if not dst.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:REFERENCECRACKCONDITIONS"):
        dst.newidfobject(
            "AIRFLOWNETWORK:MULTIZONE:REFERENCECRACKCONDITIONS",
            Name="ReferenceCrackConditions",
            Reference_Temperature=20.0,
            Reference_Barometric_Pressure=101325,
            Reference_Humidity_Ratio=0.0,
        )
    sc = dst.idfobjects.get("SCHEDULE:CONSTANT", [])
    have = {s.Name.lower() for s in sc}
    if "alwaysonschedule" not in have:
        dst.newidfobject("SCHEDULE:CONSTANT", Name="AlwaysOnSchedule",
                         Schedule_Type_Limits_Name="OnOff", Hourly_Value=1)
    if "alwaysoffschedule" not in have:
        dst.newidfobject("SCHEDULE:CONSTANT", Name="AlwaysOffSchedule",
                         Schedule_Type_Limits_Name="OnOff", Hourly_Value=0)

def get_windows_in_zone(idf, zone_name: str):
    """Return all window fenestrations belonging to a given zone."""
    zone_name_lc = zone_name.lower()

    # Step 1: all building surfaces in this zone
    zone_surfaces = [
        s for s in idf.idfobjects["BUILDINGSURFACE:DETAILED"]
        if s.Zone_Name and s.Zone_Name.lower() == zone_name_lc
    ]
    zone_surface_names = {s.Name for s in zone_surfaces}

    # Step 2: all fenestrations whose parent surface is in this zone
    windows = [
        f for f in idf.idfobjects["FENESTRATIONSURFACE:DETAILED"]
        if f.Surface_Type.lower() == "window"
        and f.Building_Surface_Name in zone_surface_names
    ]
    return windows


def _ensure_blinds(dst: IDF, zone_name: str = "bedroom_3", blind_name: str = "always_closed_blind_mat"):
    """Ensure a blind material and a shading control are attached to all windows in the given zone."""

    # 1) Add the blind material if not already present
    if not any(mat.Name.lower() == blind_name.lower() for mat in dst.idfobjects["WINDOWMATERIAL:SHADE"]):
        dst.newidfobject(
            "WINDOWMATERIAL:SHADE",
            Name=blind_name,
            Solar_Transmittance=0.2,
            Solar_Reflectance=0.2,
            Visible_Transmittance=0.05,
            Visible_Reflectance=0.3,
            Infrared_Hemispherical_Emissivity=0.9,
            Infrared_Transmittance=0.0,
            Thickness=0.9,
            Conductivity=0.1,
            Shade_to_Glass_Distance=0.1,
            Top_Opening_Multiplier=0.001,
            Bottom_Opening_Multiplier=0.002
        )

    # 2) Find all windows in the specified zone (via parent surfaces)
    windows = get_windows_in_zone(dst, zone_name)

    if not windows:
        print(f"⚠️ No windows found in zone {zone_name}, skipping blind creation.")
        return

    # 3) Create a single shading control object applying to all windows
    ctrl_name = f"{zone_name}_blinds"
    if not any(sc.Name.lower() == ctrl_name.lower() for sc in dst.idfobjects["WINDOWSHADINGCONTROL"]):
        sc = dst.newidfobject(
            "WINDOWSHADINGCONTROL",
            Name=ctrl_name,
            Zone_Name=zone_name,
            Shading_Control_Sequence_Number=1,
            Shading_Type="InteriorShade",
            Shading_Control_Type="AlwaysOn",
            Schedule_Name="AlwaysOnSchedule",
            Shading_Control_Is_Scheduled="Yes",
            Glare_Control_Is_Active="No",
            Shading_Device_Material_Name=blind_name,
            Multiple_Surface_Control_Type="Group"
        )
        # attach all windows
        for i, win in enumerate(windows, start=1):
            setattr(sc, f"Fenestration_Surface_{i}_Name", win.Name)

    print(f"✅ Added blinds ({blind_name}) to {len(windows)} window(s) in {zone_name}.")


# ----------------- main transplant -----------------
def transplant_geometry_and_afn(
    src: IDF,
    dst: IDF,
    zones: Optional[Iterable[str]] = None,
    clear_existing: bool = True,
    round_vertex_decimals: int = 6,
):
    """
    Transplant BuildingSurface + Fenestration + AFN surfaces (and AFN deps) from src -> dst.

    - Selects geometry by zone filter (if provided).
    - Removes matching geometry/AFN in dst before inserting (if clear_existing=True).
    - Copies AFN child deps referenced by those AFN surfaces.
    - Ensures AFN ReferenceCrackConditions and AlwaysOn/Off schedules exist in dst.
    """
    # 1) select building surfaces in src
    bldg_src = src.idfobjects["BUILDINGSURFACE:DETAILED"]

    if zones:
        zones_lc = {z.lower() for z in zones}
        bldg_src = [s for s in bldg_src if s.Zone_Name.lower() in zones_lc]
    bldg_names = {s.Name for s in bldg_src}

    # 2) select fenestrations attached to those
    fens_src = src.idfobjects.get("FENESTRATIONSURFACE:DETAILED", [])
    fen_src_for_parent = [f for f in fens_src if f.Building_Surface_Name in bldg_names]
    fen_names = {f.Name for f in fen_src_for_parent}

    for name in fen_names:
        if name == "Block Subfloor Storey 0 Wall 0002 window":
            print("Yess?")

    # 3) AFN surfaces that reference selected geometry
    afn_surfs_src = src.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:SURFACE", [])
    afn_for_geo = []
    for a in afn_surfs_src:
        sn = a.Surface_Name.lower()
        if sn in {n.lower() for n in bldg_names | fen_names}:
            afn_for_geo.append(a)
        else:
            # catch external or unpaired openings by zone name
            for z in (zones or []):
                if z.lower() in sn:
                    afn_for_geo.append(a)
                    break


    # 4) Gather AFN dependencies needed by those surfaces
    need_cracks, need_simple, need_horiz, need_extnodes = set(), set(), set(), set()
    for a in afn_for_geo:
        comp = a.Leakage_Component_Name
        if any(c.Name == comp for c in src.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:SURFACE:CRACK", [])):
            need_cracks.add(comp)
        elif any(c.Name == comp for c in src.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:COMPONENT:SIMPLEOPENING", [])):
            need_simple.add(comp)
        elif any(c.Name == comp for c in src.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:COMPONENT:HORIZONTALOPENING", [])):
            need_horiz.add(comp)
        en = getattr(a, "External_Node_Name", "")
        if en:
            need_extnodes.add(en)

    # ExternalNodes -> WPC Values -> CP Arrays
    nodes_src = {n.Name: n for n in src.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:EXTERNALNODE", [])}
    wpc_vals_src = {v.Name: v for v in src.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:WINDPRESSURECOEFFICIENTVALUES", [])}
    cp_arrays_src = {a.Name: a for a in src.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:WINDPRESSURECOEFFICIENTARRAY", [])}

    wpc_needed = []
    cp_arrays_needed = set()
    for nm in need_extnodes:
        node = nodes_src.get(nm)
        if node:
            curve = node.Wind_Pressure_Coefficient_Curve_Name
            if curve and curve in wpc_vals_src:
                wpc_needed.append(wpc_vals_src[curve])
                if wpc_vals_src[curve].AirflowNetworkMultiZoneWindPressureCoefficientArray_Name:
                    cp_arrays_needed.add(wpc_vals_src[curve].AirflowNetworkMultiZoneWindPressureCoefficientArray_Name)

    # 5) Clear matching in dst (AFN surfaces -> fenestrations -> building surfaces)
    if clear_existing:
        classes_to_clear = [
            "AIRFLOWNETWORK:MULTIZONE:SURFACE",
            "FENESTRATIONSURFACE:DETAILED",
            "BUILDINGSURFACE:DETAILED",
            "AIRFLOWNETWORK:MULTIZONE:SURFACE:CRACK",
            "AIRFLOWNETWORK:MULTIZONE:COMPONENT:SIMPLEOPENING",
            "AIRFLOWNETWORK:MULTIZONE:COMPONENT:HORIZONTALOPENING",
            "CONSTRUCTION",
            "MATERIAL",
            "MATERIAL:NOMASS",
            "MATERIAL:INFRAREDTRANSPARENT",
            "WINDOWMATERIAL:GLAZING",
            "WINDOWMATERIAL:SIMPLEGLAZINGSYSTEM",
            "WINDOWMATERIAL:REFRACTIVEINDEX",
            "WINDOWMATERIAL:GAS",
        ]
        for cls in classes_to_clear:
            if cls in dst.idfobjects:
                for obj in list(dst.idfobjects[cls]):
                    dst.removeidfobject(obj)


    # 6) Basics in dst
    _ensure_basics(dst)

    # 7) Copy geometry (parents then children)
    for s in bldg_src:
        clone_building_surface(dst, s, round_vertex_decimals=round_vertex_decimals)
    for f in fen_src_for_parent:
        clone_fenestration_surface(dst, f, round_vertex_decimals=round_vertex_decimals)

    # 8) Copy AFN deps
    cracks_src = {c.Name: c for c in src.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:SURFACE:CRACK", [])}
    simp_src   = {c.Name: c for c in src.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:COMPONENT:SIMPLEOPENING", [])}
    horiz_src  = {c.Name: c for c in src.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:COMPONENT:HORIZONTALOPENING", [])}

    cracks_dst = _names(dst.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:SURFACE:CRACK", []))
    simp_dst   = _names(dst.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:COMPONENT:SIMPLEOPENING", []))
    horiz_dst  = _names(dst.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:COMPONENT:HORIZONTALOPENING", []))
    for nm in sorted(need_cracks):
        if nm not in cracks_dst: _clone_obj(dst, cracks_src[nm])
    for nm in sorted(need_simple):
        if nm not in simp_dst:   _clone_obj(dst, simp_src[nm])
    for nm in sorted(need_horiz):
        if nm not in horiz_dst:  _clone_obj(dst, horiz_src[nm])

    nodes_dst = _names(dst.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:EXTERNALNODE", []))
    for nm in sorted(need_extnodes):
        if nm not in nodes_dst and nm in nodes_src:
            _clone_obj(dst, nodes_src[nm])

    wpc_dst = _names(dst.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:WINDPRESSURECOEFFICIENTVALUES", []))
    for w in wpc_needed:
        if w.Name not in wpc_dst:
            _clone_obj(dst, w)

    cp_arrays_dst = _names(dst.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:WINDPRESSURECOEFFICIENTARRAY", []))
    for nm in sorted(cp_arrays_needed):
        if nm not in cp_arrays_dst and nm in cp_arrays_src:
            _clone_obj(dst, cp_arrays_src[nm])

    # 9) Copy AFN surfaces
    for a in afn_for_geo:
        _clone_obj(dst, a)

    # 10) Preflight: report missing zones / constructions referenced by transplanted geometry
    #    (we don't auto-copy constructions/materials here to avoid hidden side-effects)
    z_needed = {s.Zone_Name for s in bldg_src}
    z_have   = {z.Name for z in dst.idfobjects["ZONE"]}
    missing_z = sorted(z_needed - z_have)

    # 10b) Copy constructions and materials from src -> dst (overwrite all)
    mat_classes = [
        "CONSTRUCTION",
        "MATERIAL",
        "MATERIAL:NOMASS",
        "MATERIAL:INFRAREDTRANSPARENT",
        "WINDOWMATERIAL:GLAZING",
        "WINDOWMATERIAL:SIMPLEGLAZINGSYSTEM",
        "WINDOWMATERIAL:REFRACTIVEINDEX",
        "WINDOWMATERIAL:GAS",
    ]
    for cls in mat_classes:
        for obj in src.idfobjects.get(cls, []):
            _clone_obj(dst, obj)

    _ensure_blinds(dst, zone_name="bedroom_3")

    cons_needed = set()
    cons_needed |= {s.Construction_Name for s in bldg_src if getattr(s, "Construction_Name", "")}
    cons_needed |= {f.Construction_Name for f in fen_src_for_parent if getattr(f, "Construction_Name", "")}
    cons_have = {c.Name for c in dst.idfobjects.get("CONSTRUCTION", [])}
    missing_c = sorted(cons_needed - cons_have)

    if missing_z:
        print("⚠️ Missing Zones in dst (referenced by transplanted surfaces):", missing_z)
    if missing_c:
        print("⚠️ Missing Constructions in dst (referenced by transplanted surfaces):", missing_c)

    print(f"✅ Transplanted {len(bldg_names)} building surfaces, "
          f"{len(fen_names)} fenestrations, {len(afn_for_geo)} AFN surfaces.")

# ----------------- example usage -----------------

# Edit these paths for your environment:
IDF.setiddname("/usr/local/EnergyPlus-9-5-0/Energy+.idd")
src_path = "/workspaces/CUBES/beizaee_validation/test.idf"            # donor (has the AFN & geometry you want)
dst_path = "/workspaces/CUBES/beizaee_validation/h28_model_afn.idf"   # recipient/base model
out_path = "/workspaces/CUBES/beizaee_validation/h28_test.idf"

src = IDF(src_path)
dst = IDF(dst_path)

# Move *all* geometry + AFN:
transplant_geometry_and_afn(src, dst,
                            zones=None,              # or e.g. ["front_room","kitchen","hall_upstairs"]
                            clear_existing=True,
                            round_vertex_decimals=6)

dst.saveas(out_path)
print(f"💾 Saved: {out_path}")


✅ Added blinds (always_closed_blind_mat) to 1 window(s) in bedroom_3.
✅ Transplanted 86 building surfaces, 29 fenestrations, 100 AFN surfaces.
💾 Saved: /workspaces/CUBES/beizaee_validation/h28_test.idf


In [5]:
from eppy import modeleditor
from eppy.modeleditor import IDF
import numpy as np


# --- CONFIGURATION ---
idf_path = "/workspaces/CUBES/beizaee_validation/test.idf"
idd_path = "/usr/local/EnergyPlus-9-5-0/Energy+.idd"

IDF.setiddname(idd_path)
idf = IDF(idf_path)


def plane_from_points(pts):
    pts = np.array(pts)
    centroid = pts.mean(axis=0)
    uu, _, vh = np.linalg.svd(pts - centroid)
    normal = vh[-1] / np.linalg.norm(vh[-1])
    return centroid, normal

def max_distance_from_plane(pts, centroid, normal):
    return np.max(np.abs(np.dot(pts - centroid, normal)))

tolerance = 1e-3  # 1 mm tolerance
warped = []

for sf in idf.idfobjects["BUILDINGSURFACE:DETAILED"]:
    coords = np.array(sf.coords)
    if len(coords) < 3:
        continue
    centroid, n = plane_from_points(coords)
    deviation = max_distance_from_plane(coords, centroid, n)
    if deviation > tolerance:
        warped.append((sf.Name, sf.Zone_Name, deviation))

print(f"\n🔍 Found {len(warped)} warped surfaces (> {tolerance*1000:.1f} mm off-plane):")
for name, zone, dev in warped:
    print(f"  • {name}  (zone={zone}, deviation={dev*1000:.2f} mm)")



🔍 Found 0 warped surfaces (> 1.0 mm off-plane):


In [6]:
from shapely.geometry import Polygon, Point, LineString
import numpy as np

def in_polygon_2d(poly, pt, tol=1e-4):
    return poly.buffer(tol).contains(pt)

def dist_to_edges(poly, pt):
    """Return distance to nearest edge and which edge it is."""
    edges = list(zip(poly.exterior.coords[:-1], poly.exterior.coords[1:]))
    dists = [LineString(e).distance(pt) for e in edges]
    idx = int(np.argmin(dists))
    return dists[idx], edges[idx]

for fs in idf.idfobjects["FENESTRATIONSURFACE:DETAILED"]:
    wall = next((s for s in idf.idfobjects["BUILDINGSURFACE:DETAILED"] if s.Name == fs.Building_Surface_Name), None)
    if not wall:
        continue
    wall_poly = Polygon([(x, y) for x, y, z in wall.coords])
    print(f"\n🧱 Checking {fs.Name} against {wall.Name}")
    for i, (x, y, z) in enumerate(fs.coords, 1):
        p = Point(x, y)
        inside = in_polygon_2d(wall_poly, p)
        d, edge = dist_to_edges(wall_poly, p)
        if not inside:
            side = "outside" if not wall_poly.touches(p) else "on edge"
            x0, y0 = edge[0]; x1, y1 = edge[1]
            print(f"⚠️ Vertex {i}: ({x:.3f},{y:.3f}) {side} wall boundary")
            print(f"   ↳ Nearest edge: ({x0:.3f},{y0:.3f})–({x1:.3f},{y1:.3f}), distance={d:.4f} m")
        else:
            print(f"✅ Vertex {i}: inside wall (dist {d:.4f} m to nearest edge)")



🧱 Checking Block front_room Storey 0 Wall 0004_window against Block front_room Storey 0 Wall 0004
⚠️ Vertex 1: (1.863,0.000) outside wall boundary
   ↳ Nearest edge: (1.800,0.000)–(5.400,0.000), distance=0.0001 m
⚠️ Vertex 2: (1.863,0.000) outside wall boundary
   ↳ Nearest edge: (1.800,0.000)–(5.400,0.000), distance=0.0001 m
⚠️ Vertex 3: (3.959,0.000) outside wall boundary
   ↳ Nearest edge: (1.800,0.000)–(5.400,0.000), distance=0.0001 m
⚠️ Vertex 4: (3.959,0.000) outside wall boundary
   ↳ Nearest edge: (1.800,0.000)–(5.400,0.000), distance=0.0001 m

🧱 Checking Block backroom Storey 0 Wall 0002_window against Block backroom Storey 0 Wall 0002
⚠️ Vertex 1: (5.345,8.100) outside wall boundary
   ↳ Nearest edge: (5.400,8.100)–(2.700,8.100), distance=0.0001 m
⚠️ Vertex 2: (5.345,8.100) outside wall boundary
   ↳ Nearest edge: (5.400,8.100)–(2.700,8.100), distance=0.0001 m
⚠️ Vertex 3: (3.684,8.100) outside wall boundary
   ↳ Nearest edge: (5.400,8.100)–(2.700,8.100), distance=0.0001 m
⚠

/usr/local/lib/python3.9/dist-packages/shapely/measurement.py:72: RuntimeWarning: invalid value encountered in distance
  return lib.distance(a, b, **kwargs)


In [7]:
from eppy.modeleditor import IDF

IDF.setiddname("/usr/local/EnergyPlus-9-5-0/Energy+.idd")
src_path = "/workspaces/CUBES/beizaee_validation/h28_test.idf"
out_path = "/workspaces/CUBES/beizaee_validation/h28_coheat.idf"

idf = IDF(src_path)

# --- 1) Remove HVAC and plant-related objects ---
remove_classes = [
    "PLANTLOOP","AIRLOOPHVAC","BRANCH","BRANCHLIST","CONNECTOR:MIXER","CONNECTOR:SPLITTER",
    "CONNECTORLIST","NODELIST","PUMP:*","BOILER:*","CHILLER:*","COIL:*","AIRTERMINAL:*",
    "ZONEHVAC:*","FAN:*","SIZING:SYSTEM","SIZING:PLANT","CONTROLLER:*","SETPOINTMANAGER:*",
    "HVACTEMPLATE:*"
]
for key in list(idf.idfobjects.keys()):
    if any(key.upper().startswith(cls.split(":")[0]) for cls in remove_classes):
        for obj in list(idf.idfobjects[key]):
            idf.removeidfobject(obj)

# --- Remove all thermostat-related controls to prevent duplicates ---
for cls in [
    "ZONECONTROL:THERMOSTAT",
    "THERMOSTATSETPOINT:SINGLEHEATING",
    "THERMOSTATSETPOINT:SINGLECOOLING",
    "THERMOSTATSETPOINT:DUALSETPOINT"
]:
    for obj in list(idf.idfobjects.get(cls, [])):
        idf.removeidfobject(obj)

# --- Remove boiler, plant, pipe, and EMS controls completely ---
extra_remove = [
    "PIPE:*","PLANTEQUIPMENTLIST","PLANTEQUIPMENTOPERATION:*",
    "PLANTEQUIPMENTOPERATIONSCHEMES","ENERGYMANAGEMENTSYSTEM:*","CURVE:*"
]
for key in list(idf.idfobjects.keys()):
    if any(key.upper().startswith(cls.split(":")[0]) for cls in extra_remove):
        for obj in list(idf.idfobjects[key]):
            idf.removeidfobject(obj)

# --- 2) Always-on/off schedules ---
if not any(s.Name.lower()=="alwaysonschedule" for s in idf.idfobjects.get("SCHEDULE:CONSTANT",[])):
    idf.newidfobject("SCHEDULE:CONSTANT",Name="AlwaysOnSchedule",Schedule_Type_Limits_Name="OnOff",Hourly_Value=1)
if not any(s.Name.lower()=="alwaysoffschedule" for s in idf.idfobjects.get("SCHEDULE:CONSTANT",[])):
    idf.newidfobject("SCHEDULE:CONSTANT",Name="AlwaysOffSchedule",Schedule_Type_Limits_Name="OnOff",Hourly_Value=0)

# --- 3) Replace RunPeriod with co-heating test window ---
for rp in list(idf.idfobjects.get("RUNPERIOD", [])):
    idf.removeidfobject(rp)
idf.newidfobject("RUNPERIOD",
                 Name="Coheat_Test_Period",
                 Begin_Month=11, Begin_Day_of_Month=23, Begin_Year=2013,
                 End_Month=12, End_Day_of_Month=1, End_Year=2013)

# --- 4) Ensure all AFN surfaces use AlwaysOnSchedule ---
for surf in idf.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:SURFACE", []):
    if "door-schedule" in surf.Venting_Availability_Schedule_Name:
        surf.Venting_Availability_Schedule_Name = "AlwaysOnSchedule"

# --- 5) Zone heating setpoints (°C) ---
temps = {
    "front_room": 24.32,
    "backroom": 24.64,
    "kitchen": 25.15,
    "hall_downstairs": 24.0,
    "hall_upstairs": 24.67,
    "bedroom_1": 24.62,
    "bedroom_2": 24.67,
    "bathroom": 23.53,
    "bedroom_3": 24.83,
}

# --- 6) Create per-zone thermostat schedules and ideal loads systems ---
excluded = {"loft", "subfloor"}

for zone in idf.idfobjects["ZONE"]:
    zname = zone.Name.lower()
    if any(ex in zname for ex in excluded):
        print(f"Skipping unconditioned zone: {zone.Name}")
        continue

    if zname in temps:
        sched_name = f"{zone.Name}-thermostat dual sp control-heating-ext"
        idf.newidfobject("SCHEDULE:COMPACT",
                         Name=sched_name,
                         Schedule_Type_Limits_Name="Temperature",
                         Field_1="Through: 12/31",
                         Field_2="For: AllDays",
                         Field_3="Until: 24:00",
                         Field_4=str(temps[zname]))

        sp_name = f"{zone.Name}_Coheat_Set"
        idf.newidfobject("THERMOSTATSETPOINT:SINGLEHEATING",
                         Name=sp_name,
                         Setpoint_Temperature_Schedule_Name=sched_name)

        tstat_name = f"{zone.Name}_Thermostat"
        idf.newidfobject("ZONECONTROL:THERMOSTAT",
                         Name=tstat_name,
                         Zone_or_ZoneList_Name=zone.Name,
                         Control_Type_Schedule_Name="AlwaysOnSchedule",
                         Control_1_Object_Type="ThermostatSetpoint:SingleHeating",
                         Control_1_Name=sp_name)

    sys_name=f"{zone.Name}_IdealLoads"
    idf.newidfobject("ZONEHVAC:IDEALLOADSAIRSYSTEM",
                     Name=sys_name,
                     Availability_Schedule_Name="AlwaysOnSchedule",
                     Zone_Supply_Air_Node_Name=f"{zone.Name}_Supply",
                     Zone_Exhaust_Air_Node_Name=f"{zone.Name}_Exhaust",
                     Maximum_Heating_Supply_Air_Temperature=50,
                     Minimum_Cooling_Supply_Air_Temperature=13,
                     Maximum_Heating_Supply_Air_Humidity_Ratio=0.0156,
                     Minimum_Cooling_Supply_Air_Humidity_Ratio=0.0077,
                     Heating_Limit="NoLimit",
                     Cooling_Limit="NoLimit",
                     Heating_Availability_Schedule_Name="AlwaysOnSchedule",
                     Cooling_Availability_Schedule_Name="AlwaysOffSchedule",
                     Dehumidification_Control_Type="None",
                     Cooling_Sensible_Heat_Ratio=0.7,
                     Humidification_Control_Type="None",
                     Design_Specification_Outdoor_Air_Object_Name="",
                     Outdoor_Air_Inlet_Node_Name="",
                     Demand_Controlled_Ventilation_Type="None",
                     Outdoor_Air_Economizer_Type="NoEconomizer",
                     Heat_Recovery_Type="None",
                     Sensible_Heat_Recovery_Effectiveness=0.0,
                     Latent_Heat_Recovery_Effectiveness=0.0)

    eq_list=f"{zone.Name}_EquipList"
    idf.newidfobject("ZONEHVAC:EQUIPMENTLIST",
                     Name=eq_list,
                     Load_Distribution_Scheme="SequentialLoad",
                     Zone_Equipment_1_Object_Type="ZoneHVAC:IdealLoadsAirSystem",
                     Zone_Equipment_1_Name=sys_name,
                     Zone_Equipment_1_Cooling_Sequence=1,
                     Zone_Equipment_1_Heating_or_NoLoad_Sequence=1,
                     Zone_Equipment_1_Sequential_Cooling_Fraction_Schedule_Name="",
                     Zone_Equipment_1_Sequential_Heating_Fraction_Schedule_Name="")

    idf.newidfobject("ZONEHVAC:EQUIPMENTCONNECTIONS",
                     Zone_Name=zone.Name,
                     Zone_Conditioning_Equipment_List_Name=eq_list,
                     Zone_Air_Inlet_Node_or_NodeList_Name=f"{zone.Name}_Supply",
                     Zone_Air_Exhaust_Node_or_NodeList_Name=f"{zone.Name}_Exhaust",
                     Zone_Air_Node_Name=f"{zone.Name}_AirNode",
                     Zone_Return_Air_Node_or_NodeList_Name=f"{zone.Name}_Return")

# --- 7) Add output variables for co-heating analysis ---
idf.newidfobject("OUTPUT:VARIABLE",
                 Key_Value="*",
                 Variable_Name="Zone Ideal Loads Supply Air Total Heating Energy",
                 Reporting_Frequency="TimeStep")
idf.newidfobject("OUTPUT:VARIABLE",
                 Key_Value="*",
                 Variable_Name="Zone Mean Air Temperature",
                 Reporting_Frequency="TimeStep")
idf.newidfobject("OUTPUT:VARIABLE",
                 Key_Value="*",
                 Variable_Name="Zone Air System Sensible Heating Rate",
                 Reporting_Frequency="TimeStep")

# --- 8) Update SimulationControl ---
for sc in list(idf.idfobjects.get("SIMULATIONCONTROL", [])):
    idf.removeidfobject(sc)

idf.newidfobject("SIMULATIONCONTROL",
                 Do_Zone_Sizing_Calculation="No",
                 Do_System_Sizing_Calculation="No",
                 Do_Plant_Sizing_Calculation="No",
                 Run_Simulation_for_Sizing_Periods="No",
                 Run_Simulation_for_Weather_File_Run_Periods="Yes",
                 Do_HVAC_Sizing_Simulation_for_Sizing_Periods="No",
                 Maximum_Number_of_HVAC_Sizing_Simulation_Passes=1)

# --- 9) Normalise AirflowNetwork names to match geometry ---
# Build dictionaries of canonical names (case-insensitive lookup)
zone_names={z.Name.lower():z.Name for z in idf.idfobjects["ZONE"]}
surf_names={s.Name.lower():s.Name for s in idf.idfobjects["BUILDINGSURFACE:DETAILED"]}

# Fix AFN:Zone zone names
for afn_zone in idf.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:ZONE",[]):
    name=afn_zone.Zone_Name.lower()
    if name in zone_names: afn_zone.Zone_Name=zone_names[name]

# Fix AFN:Surface surface names and linked external nodes
for afn_surf in idf.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:SURFACE",[]):
    sname=afn_surf.Surface_Name.lower()
    if sname in surf_names: afn_surf.Surface_Name=surf_names[sname]
    zname=afn_surf.Zone_Name.lower() if hasattr(afn_surf,"Zone_Name") else None
    if zname and zname in zone_names: afn_surf.Zone_Name=zone_names[zname]

# Fix AFN:ExternalNode names consistency with surfaces
ext_nodes={n.Name.lower():n for n in idf.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:EXTERNALNODE",[])}
for afn_surf in idf.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:SURFACE",[]):
    ename=afn_surf.External_Node_Name.lower()
    if ename in ext_nodes: afn_surf.External_Node_Name=ext_nodes[ename].Name


# Add AFN linkage output variables
idf.newidfobject(
    "OUTPUT:VARIABLE",
    Key_Value="*",
    Variable_Name="AFN Linkage Node 1 to Node 2 Mass Flow Rate",
    Reporting_Frequency="Timestep"
)

idf.newidfobject(
    "OUTPUT:VARIABLE",
    Key_Value="*",
    Variable_Name="AFN Linkage Node 2 to Node 1 Mass Flow Rate",
    Reporting_Frequency="Timestep"
)


idf.saveas(out_path)
print(f"✅ Final co-heating model saved with energy outputs: {out_path}")


Skipping unconditioned zone: Subfloor
Skipping unconditioned zone: Loft
✅ Final co-heating model saved with energy outputs: /workspaces/CUBES/beizaee_validation/h28_coheat.idf


In [8]:
# from typing import Optional, Tuple, Dict
# from eppy.modeleditor import IDF

# def scale_crack_coefficients_by_zone(
#     idf: IDF,
#     zone_factors: Dict[str, float],
#     only_outdoors: bool = True,
#     exclude_name_prefixes: Tuple[str, ...] = ("AFN_Tiny",),
#     min_coef: Optional[float] = None,
#     max_coef: Optional[float] = None,
#     new_exponent: Optional[float] = None,
# ) -> int:
#     """
#     Scale AFN crack coefficients zone by zone.

#     Args:
#         idf: an Eppy IDF object
#         zone_factors: mapping {zone_name (case-insensitive): scale_factor}
#         only_outdoors: if True, only scale cracks on outdoor surfaces
#         exclude_name_prefixes: skip cracks whose names start with these
#         min_coef, max_coef: clip coefficient range if set
#         new_exponent: optionally override all crack exponents
#     """
#     cracks = {c.Name: c for c in idf.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:SURFACE:CRACK", [])}
#     if not cracks:
#         print("No AFN crack components found.")
#         return 0

#     to_touch = set(cracks.keys())

#     # Exclude helper cracks by prefix
#     if exclude_name_prefixes:
#         to_touch = {n for n in to_touch if not any(n.startswith(p) for p in exclude_name_prefixes)}

#     b_by_name = {s.Name: s for s in idf.idfobjects["BUILDINGSURFACE:DETAILED"]}
#     f_by_name = {f.Name: f for f in idf.idfobjects.get("FENESTRATIONSURFACE:DETAILED", [])}

#     def get_zone_and_bc(surface_name: str):
#         """Return (zone_name_lower, boundary_condition_lower)."""
#         surf = b_by_name.get(surface_name)
#         if surf is not None:
#             return surf.Zone_Name.lower(), (surf.Outside_Boundary_Condition or "").lower()
#         fen = f_by_name.get(surface_name)
#         if fen is not None:
#             parent = b_by_name.get(fen.Building_Surface_Name)
#             if parent:
#                 return parent.Zone_Name.lower(), (parent.Outside_Boundary_Condition or "").lower()
#         return "", ""

#     # Assign scaling factors
#     cracks_to_scale: Dict[str, float] = {}
#     for afn in idf.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:SURFACE", []):
#         comp = afn.Leakage_Component_Name
#         if comp not in cracks or comp not in to_touch:
#             continue
#         zone, bc = get_zone_and_bc(afn.Surface_Name)
#         if only_outdoors and bc != "outdoors":
#             continue
#         factor = zone_factors.get(zone, 1.0)  # default: no scaling
#         if factor != 1.0:
#             cracks_to_scale[comp] = factor

#     # Apply scaling
#     changed = 0
#     for name, factor in cracks_to_scale.items():
#         c = cracks[name]
#         try:
#             old = float(c.Air_Mass_Flow_Coefficient_at_Reference_Conditions)
#         except Exception:
#             print(f"Skipping {name}: cannot read coefficient.")
#             continue

#         new = old * factor
#         if min_coef is not None:
#             new = max(new, float(min_coef))
#         if max_coef is not None:
#             new = min(new, float(max_coef))
#         if new <= 0:
#             print(f"Skipping {name}: computed non-positive coef {new}")
#             continue

#         c.Air_Mass_Flow_Coefficient_at_Reference_Conditions = new

#         if new_exponent is not None:
#             try:
#                 c.Air_Mass_Flow_Exponent = float(new_exponent)
#             except Exception:
#                 pass

#         changed += 1
#         # print(f"{name}: {old:.4g} -> {new:.4g} (factor {factor})")

#     print(f"Updated {changed} crack component(s).")
#     return changed


# IDF.setiddname("/usr/local/EnergyPlus-9-5-0/Energy+.idd")
# idf = IDF("/workspaces/CUBES/beizaee_validation/h28_test.idf")

# zone_factors = {'front_room': 4.397134777338084,
#                 'backroom': 15.984065581845977 * 0.6,
#                 'kitchen': 32.844684736398214 * 0.7,
#                 'hall_downstairs': 22.59798236724478,
#                 'hall_upstairs': 65.86109469156875,
#                 'bedroom_3': 69.74240211937318*200,
#                 'bathroom': 9.534381803768378
#                 }

# zone_factor_scale = 0.05

# # Scale all values in the dict
# zone_factors = {z: v * zone_factor_scale for z, v in zone_factors.items()}

# scale_crack_coefficients_by_zone(
#     idf,
#     zone_factors=zone_factors,
#     only_outdoors=True,
#     exclude_name_prefixes=("Block Subfloor Storey 0 Wall",)
# )

# idf.save()


In [9]:
# from eppy.modeleditor import IDF
# from pathlib import Path

# # --- Paths ---
# EPLUS_PATH = "/usr/local/EnergyPlus-9-5-0"
# iddfile = Path(EPLUS_PATH) / "Energy+.idd"
# idf_file = "/workspaces/CUBES/beizaee_validation/h28_model_afn_adiabatic.idf"
# idf_out  = "/workspaces/CUBES/beizaee_validation/h28_model_afn_adiabatic.idf"

# IDF.setiddname(str(iddfile))
# idf = IDF(idf_file)

# # --- Config ---
# ADIABATIC_CONSTRUCTION = "Adiabatic Wall-Construction"
# TARGET_AZIMUTHS = [90.0]   # edit to the party-wall orientation(s)
# AZ_TOL = 1e-2

# def get_azimuth(surf):
#     # eppy typically exposes computed azimuth; fallbacks in case of version differences
#     for attr in ("Azimuth_Angle", "azimuth"):
#         if hasattr(surf, attr):
#             try:
#                 return float(getattr(surf, attr))
#             except Exception:
#                 pass
#     raise AttributeError(f"No azimuth attribute available for surface '{surf.Name}'")

# converted_names = []

# # --- Convert qualifying external walls to Adiabatic ---
# for s in list(idf.idfobjects["BUILDINGSURFACE:DETAILED"]):
#     if s.Surface_Type.lower() != "wall":
#         continue
#     if s.Outside_Boundary_Condition.lower() != "outdoors":
#         continue

#     az = get_azimuth(s)
#     if any(abs(az - t) <= AZ_TOL for t in TARGET_AZIMUTHS):
#         print(f"Converting -> {getattr(s, 'Name', '<unnamed>')} (az={az:.3f}°)")
#         s.Construction_Name = ADIABATIC_CONSTRUCTION
#         s.Outside_Boundary_Condition = "Adiabatic"
#         s.Outside_Boundary_Condition_Object = ""  # must be blank for Adiabatic
#         s.Sun_Exposure  = "NoSun"
#         s.Wind_Exposure = "NoWind"
#         converted_names.append(s.Name)

# # --- Remove AFN surfaces that point to those walls ---
# afn_key = "AIRFLOWNETWORK:MULTIZONE:SURFACE"
# if afn_key in idf.idfobjects and converted_names:
#     to_delete = [obj for obj in idf.idfobjects[afn_key]
#                  if obj.Surface_Name in converted_names]
#     for obj in to_delete:
#         print(f"Removing AFN surface for '{obj.Surface_Name}' "
#               f"(component={obj.Leakage_Component_Name})")
#         idf.removeidfobject(obj)

#     # (Optional) prune orphan external nodes referenced by nothing
#     # Build set of still-referenced external nodes
#     still_refs = set(o.External_Node_Name for o in idf.idfobjects[afn_key]) if afn_key in idf.idfobjects else set()
#     extnode_key = "AIRFLOWNETWORK:MULTIZONE:EXTERNALNODE"
#     if extnode_key in idf.idfobjects:
#         orphans = [n for n in idf.idfobjects[extnode_key] if n.Name not in still_refs]
#         for n in orphans:
#             print(f"Pruning orphan AFN external node '{n.Name}'")
#             idf.removeidfobject(n)

# idf.saveas(idf_out)
# print(f"✅ Saved: {idf_out}")
# print(f"Converted {len(converted_names)} wall(s) to Adiabatic.")


In [10]:
# from eppy.modeleditor import IDF

# IDF.setiddname("/usr/local/EnergyPlus-9-5-0/Energy+.idd")

# def afn_outdoor_summary(idf_path):
#     idf = IDF(idf_path)
#     bsurfs = {s.Name: s for s in idf.idfobjects["BUILDINGSURFACE:DETAILED"]}
#     afn = idf.idfobjects.get("AIRFLOWNETWORK:MULTIZONE:SURFACE", [])
#     by_zone = {}

#     for a in afn:
#         parent = bsurfs.get(a.Surface_Name)
#         if not parent:
#             continue
#         z = parent.Zone_Name
#         bc = (parent.Outside_Boundary_Condition or "").lower()
#         has_ext = bool(a.External_Node_Name)
#         rec = by_zone.setdefault(z, {"outdoors":0,"internal":0,"missing_extnode":[],"examples":[]})
#         if bc == "outdoors":
#             rec["outdoors"] += 1
#             if not has_ext:
#                 rec["missing_extnode"].append(a.Surface_Name)
#         else:
#             rec["internal"] += 1
#         rec["examples"].append((a.Surface_Name, bc, a.External_Node_Name or ""))

#     for z in sorted(by_zone):
#         r = by_zone[z]
#         print(f"{z:15s}  outdoors={r['outdoors']:2d}  internal={r['internal']:2d}"
#               + (f"  MISSING extnode: {r['missing_extnode']}" if r['missing_extnode'] else ""))
#     return by_zone

# # Example:
# by_zone = afn_outdoor_summary("/workspaces/CUBES/beizaee_validation/h28_model_afn_adiabatic.idf")


In [11]:
# import os
# from pathlib import Path
# from eppy import modeleditor
# from eppy.modeleditor import IDF

# # Define EnergyPlus installation path
# EPLUS_PATH = "/usr/local/EnergyPlus-9-5-0/"  # Update if needed

# # Set the IDD file path
# iddfile = os.path.join(EPLUS_PATH, "Energy+.idd")
# IDF.setiddname(iddfile)  # Load the IDD file

# idf_file = "/workspaces/CUBES/beizaee_validation/h28_model_afn_adiabatic.idf"

# # Define the weather file path
# weather_file_path = "/workspaces/CUBES/cubes/data/weather/loughborough.epw"

# # ✅ Initialize IDF with both the IDF file and Weather file
# idf = IDF(idf_file, weather_file_path)

# # Print IDF summary
# print("IDF Version:", idf.idfobjects["VERSION"])
# print("Number of objects:", len(idf.idfobjects))

# # Define the test function to run the simulation
# def test_idf(idf):
#     # Define paths for saving and output
#     cwd_path = os.getcwd()
#     env_data_path = os.path.join(cwd_path, "input_case_1")
#     Path(env_data_path).mkdir(parents=True, exist_ok=True)

#     # Save the modified IDF
#     idf.save(filename=os.path.join(env_data_path, "test1.idf"))

#     # Define output directory
#     output_directory = os.path.join(env_data_path, "output/")
#     Path(output_directory).mkdir(parents=True, exist_ok=True)

#     # ✅ Run EnergyPlus simulation (No need to pass weather file separately now)
#     idf.run(output_directory=output_directory)

# # Run the test function
# test_idf(idf)


In [12]:
# idf.translate([0, 0, 15])

In [13]:
# import numpy as np
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# def edges_from_coords(coords):
#     """Return sorted edges for a surface given its coords."""
#     edges = []
#     for i in range(len(coords)):
#         p1 = tuple(np.round(coords[i], 6))  # rounding for numerical stability
#         p2 = tuple(np.round(coords[(i + 1) % len(coords)], 6))
#         edges.append(tuple(sorted((p1, p2))))
#     return edges

# def find_problem_surfaces(idf):
#     """Find surfaces with edges shared by >2 or by no other surfaces."""
#     edge_map = {}
#     for sf in idf.idfobjects["BUILDINGSURFACE:DETAILED"]:
#         for edge in edges_from_coords(sf.coords):
#             edge_map.setdefault(edge, []).append(sf.Name)

#     problem_surfaces = set()
#     for edge, surfaces in edge_map.items():
#         if len(surfaces) > 2:
#             problem_surfaces.update(surfaces)
#         # edge only used by 1 surface, but is not ground or outdoors
#         elif len(surfaces) == 1:
#             sf = next(s for s in idf.idfobjects["BUILDINGSURFACE:DETAILED"] if s.Name == surfaces[0])
#             if sf.Outside_Boundary_Condition.lower() == "surface":
#                 problem_surfaces.add(sf.Name)

#     return list(problem_surfaces)

# def plot_surfaces_3d(idf, surface_names):
#     """Plot given surfaces in 3D for debugging."""
#     surfaces = [sf for sf in idf.idfobjects["BUILDINGSURFACE:DETAILED"] if sf.Name in surface_names]
#     if not surfaces:
#         print("⚠️ No surfaces to plot.")
#         return

#     fig = plt.figure()
#     ax = fig.add_subplot(111, projection='3d')

#     for sf in surfaces:
#         verts = [sf.coords]
#         poly = Poly3DCollection(verts, alpha=0.6)
#         ax.add_collection3d(poly)
#         centroid = tuple(sum(c[j] for c in sf.coords) / len(sf.coords) for j in range(3))
#         ax.text(*centroid, sf.Name, fontsize=8)

#     # auto scale
#     all_points = [pt for sf in surfaces for pt in sf.coords]
#     xs, ys, zs = zip(*all_points)
#     ax.set_xlim(min(xs), max(xs))
#     ax.set_ylim(min(ys), max(ys))
#     ax.set_zlim(min(zs), max(zs))
#     ax.set_xlabel('X (m)')
#     ax.set_ylabel('Y (m)')
#     ax.set_zlabel('Z (m)')
#     plt.title("Problem Surfaces")
#     plt.show()


In [14]:
# problems = find_problem_surfaces(idf)
# print("Problem surfaces:", problems)

# plot_surfaces_3d(idf, problems)


In [15]:
# import matplotlib.pyplot as plt

# def plot_idf_surfaces(idf, surface_names, colors=None, show_labels=True):
#     """
#     Plot selected IDF surfaces in 2D (top-down XY projection).

#     Args:
#         idf: The IDF object.
#         surface_names (list[str]): List of surface names to plot.
#         colors (list[str], optional): List of matplotlib colors to use.
#                                       Defaults to auto color cycle.
#         show_labels (bool): Whether to label surfaces by name.
#     """
#     surfaces = [
#         sf for sf in idf.idfobjects["BUILDINGSURFACE:DETAILED"]
#         if sf.Name in surface_names
#     ]

#     if colors is None:
#         colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

#     plt.figure()
#     for i, sf in enumerate(surfaces):
#         coords = sf.coords + [sf.coords[0]]  # close the polygon
#         xs, ys, zs = zip(*coords)
#         plt.plot(xs, ys, color=colors[i % len(colors)], label=sf.Name)
#         if show_labels:
#             plt.text(sum(xs)/len(xs), sum(ys)/len(ys), sf.Name, fontsize=8)

#     plt.axis('equal')
#     plt.xlabel("X (m)")
#     plt.ylabel("Y (m)")
#     plt.title("Selected IDF Surfaces")
#     plt.legend()
#     plt.show()


In [16]:
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# def plot_idf_surfaces_3d(idf, surface_names, colors=None, show_labels=True):
#     """
#     Plot selected IDF surfaces in 3D.

#     Args:
#         idf: The IDF object.
#         surface_names (list[str]): List of surface names to plot.
#         colors (list[str], optional): Colors for each surface.
#         show_labels (bool): Whether to show labels at centroid.
#     """
#     surfaces = [
#         sf for sf in idf.idfobjects["BUILDINGSURFACE:DETAILED"]
#         if sf.Name in surface_names
#     ]
#     if not surfaces:
#         print("⚠️ No surfaces found with given names.")
#         return

#     if colors is None:
#         colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

#     fig = plt.figure()
#     ax = fig.add_subplot(111, projection='3d')

#     for i, sf in enumerate(surfaces):
#         coords = sf.coords  # list of (x, y, z)
#         verts = [coords]
#         poly = Poly3DCollection(verts, alpha=0.6, facecolor=colors[i % len(colors)])
#         ax.add_collection3d(poly)

#         if show_labels:
#             centroid = tuple(sum(c[j] for c in coords) / len(coords) for j in range(3))
#             ax.text(*centroid, sf.Name, fontsize=8)

#     # Auto scale axes
#     all_points = [pt for sf in surfaces for pt in sf.coords]
#     xs, ys, zs = zip(*all_points)
#     ax.set_xlim(min(xs), max(xs))
#     ax.set_ylim(min(ys), max(ys))
#     ax.set_zlim(min(zs), max(zs))

#     ax.set_xlabel('X (m)')
#     ax.set_ylabel('Y (m)')
#     ax.set_zlabel('Z (m)')
#     plt.title("3D IDF Surfaces")
#     plt.show()


In [17]:
# plot_idf_surfaces_3d(idf, [
#     "Block hall_downstairs Storey 0 Wall 0002_1",
#     "Block kitchen Storey 0 Wall 0004_1"
# ])


In [18]:
# plot_idf_surfaces(idf, ["Block front_room Storey 0 Wall 0002_2",
#                         "Block backroom Storey 0 Wall 0004_1"])


In [19]:
# doors = {}
# for sf in idf.idfobjects["FENESTRATIONSURFACE:DETAILED"]:
#     if "door" in sf.Name.lower():
#         doors[sf.Name] = sf.area

# holes = {}
# for sf in idf.idfobjects["FENESTRATIONSURFACE:DETAILED"]:
#     if "hole" in sf.Name.lower():
#         holes[sf.Name] = sf.area


#### Adding new materials to pickle

In [20]:
# import pickle
# import pandas as pd
# from cubes.construct.material import Material, InfraredTransparentMaterial
# materials = pd.read_pickle("/workspaces/CUBES/cubes/materials.pickle")

# door_material = Material(
#     name="Door_WoodSolid_test_35mm",   # new clear name
#     rho=700.0,                    # Density [kg/m3]
#     cp=2390.0,                    # Specific Heat [J/kg-K]
#     k=0.19,                       # Conductivity [W/m-K]
#     roughness="Rough",            # Surface roughness
#     thermal_absorptance=0.9,
#     solar_absorptance=0.5,
#     visual_absorptance=0.5
# )

# # Then add it to your materials dict
# materials["Door_WoodSolid_35mm"] = door_material

# materials["IRTMaterial"] = InfraredTransparentMaterial(name="IRTMaterial")


# # Save to pickle file
# path = "/workspaces/CUBES/cubes/materials.pickle"
# with open(path, "wb") as f:
#     pickle.dump(materials, f)



In [21]:

# from cubes.construct.material import Material, WindowMaterialGlazing
# windows = pd.read_pickle("/workspaces/CUBES/cubes/windows.pickle")

# windows["beizaee_glazing"] = WindowMaterialGlazing(
#     name="beizaee_glazing",
#     optical_data_type="SpectralAverage",  # no spectral dataset, use averaged values
#     data_set_name="",
#     thickness=0.003,                      # 3 mm clear glass
#     solar_transmittance=0.837,
#     front_side_solar_reflectance=0.075,
#     back_side_solar_reflectance=0.075,
#     visible_transmittance=0.898,
#     front_side_visible_reflectance=0.081,
#     back_side_visible_reflectance=0.081,
#     infrared_transmittance=0.0,           # opaque to longwave IR
#     front_side_infrared_emissivity=0.84,
#     back_side_infrared_emissivity=0.84,
#     conductivity=0.9,
# )


# # Optionally save back
# import pickle

# with open("/workspaces/CUBES/cubes/windows.pickle", "wb") as f:
#     pickle.dump(windows, f)


In [22]:
# import pandas as pd
# materials = pd.read_pickle("/workspaces/CUBES/cubes/materials.pickle")

In [23]:
# import pandas as pd
# materials = pd.read_pickle("/workspaces/CUBES/cubes/materials.pickle")


# from cubes.construct.material import Material, WindowMaterialGlazing

# # --- Beizaee opaque materials ---
# beizaee_materials = {
#     "beizaee_brick_outer_leaf": Material(
#         name="beizaee_brick_outer_leaf",
#         rho=800,
#         cp=1700,
#         k=0.84,
#         roughness="Rough",
#         thermal_absorptance=0.9,
#         solar_absorptance=0.7,
#         visual_absorptance=0.7,
#     ),
#     "beizaee_brick_inner_leaf": Material(
#         name="beizaee_brick_inner_leaf",
#         rho=800,
#         cp=1700,
#         k=0.62,
#         roughness="Rough",
#         thermal_absorptance=0.9,
#         solar_absorptance=0.7,
#         visual_absorptance=0.7,
#     ),
#     "beizaee_plaster_dense": Material(
#         name="beizaee_plaster_dense",
#         rho=1000,
#         cp=1300,
#         k=0.50,
#         roughness="Smooth",
#         thermal_absorptance=0.9,
#         solar_absorptance=0.7,
#         visual_absorptance=0.7,
#     ),
#     "beizaee_clay_tile": Material(
#         name="beizaee_clay_tile",
#         rho=800,
#         cp=2000,
#         k=1.00,
#         roughness="MediumSmooth",
#         thermal_absorptance=0.9,
#         solar_absorptance=0.7,
#         visual_absorptance=0.7,
#     ),
#     "beizaee_roofing_felt": Material(
#         name="beizaee_roofing_felt",
#         rho=837,
#         cp=960,
#         k=0.19,
#         roughness="Rough",
#         thermal_absorptance=0.9,
#         solar_absorptance=0.7,
#         visual_absorptance=0.7,
#     ),
#     "beizaee_polyisocyanurate": Material(
#         name="beizaee_polyisocyanurate",
#         rho=1470,
#         cp=45,
#         k=0.022,
#         roughness="Rough",
#         thermal_absorptance=0.9,
#         solar_absorptance=0.7,
#         visual_absorptance=0.7,
#     ),
#     "beizaee_timber_flooring": Material(
#         name="beizaee_timber_flooring",
#         rho=1200,
#         cp=650,
#         k=0.14,
#         roughness="MediumSmooth",
#         thermal_absorptance=0.9,
#         solar_absorptance=0.7,
#         visual_absorptance=0.7,
#     ),
#     "beizaee_cast_concrete": Material(
#         name="beizaee_cast_concrete",
#         rho=1000,
#         cp=2000,
#         k=1.13,
#         roughness="Rough",
#         thermal_absorptance=0.9,
#         solar_absorptance=0.7,
#         visual_absorptance=0.7,
#     ),
#     "beizaee_carpet": Material(
#         name="beizaee_carpet",
#         rho=1300,
#         cp=200,
#         k=0.06,
#         roughness="Rough",
#         thermal_absorptance=0.95,
#         solar_absorptance=0.7,
#         visual_absorptance=0.7,
#     ),
#     "beizaee_plasterboard": Material(
#         name="beizaee_plasterboard",
#         rho=896,
#         cp=2800,
#         k=0.25,
#         roughness="Smooth",
#         thermal_absorptance=0.9,
#         solar_absorptance=0.7,
#         visual_absorptance=0.7,
#     ),
#     "beizaee_painted_oak": Material(
#         name="beizaee_painted_oak",
#         rho=2390,
#         cp=700,
#         k=0.19,
#         roughness="Smooth",
#         thermal_absorptance=0.9,
#         solar_absorptance=0.7,
#         visual_absorptance=0.7,
#     ),
# }


# windows["beizaee_glazing"] = WindowMaterialGlazing(
#     name="beizaee_glazing",
#     optical_data_type="SpectralAverage",  # no spectral dataset, use averaged values
#     data_set_name="",
#     thickness=0.003,                      # 3 mm clear glass
#     solar_transmittance=0.837,
#     front_side_solar_reflectance=0.075,
#     back_side_solar_reflectance=0.075,
#     visible_transmittance=0.898,
#     front_side_visible_reflectance=0.081,
#     back_side_visible_reflectance=0.081,
#     infrared_transmittance=0.0,           # opaque to longwave IR
#     front_side_infrared_emissivity=0.84,
#     back_side_infrared_emissivity=0.84,
#     conductivity=0.9,
# )


# # --- Merge into your existing dict ---
# materials.update(beizaee_materials)

# # Optionally save back
# import pickle

# with open("/workspaces/CUBES/cubes/materials.pickle", "wb") as f:
#     pickle.dump(materials, f)



# materials = {
#     "Brick_outer_leaf": {
#         "conductivity_W_mK": 0.84,
#         "density_kg_m3": 800,
#         "specific_heat_J_kgK": 1700,
#     },
#     "Brick_inner_leaf": {
#         "conductivity_W_mK": 0.62,
#         "density_kg_m3": 800,
#         "specific_heat_J_kgK": 1700,
#     },
#     "Plaster_dense": {
#         "conductivity_W_mK": 0.50,
#         "density_kg_m3": 1000,
#         "specific_heat_J_kgK": 1300,
#     },
#     "Clay_tile": {
#         "conductivity_W_mK": 1.00,
#         "density_kg_m3": 800,
#         "specific_heat_J_kgK": 2000,
#     },
#     "Roofing_felt": {
#         "conductivity_W_mK": 0.19,
#         "density_kg_m3": 837,
#         "specific_heat_J_kgK": 960,
#     },
#     "Glazing": {
#         "conductivity_W_mK": 0.90,
#         "density_kg_m3": None,
#         "specific_heat_J_kgK": None,
#     },
#     "Polyisocyanurate": {
#         "conductivity_W_mK": 0.022,
#         "density_kg_m3": 1470,
#         "specific_heat_J_kgK": 45,
#     },
#     "Timber_flooring": {
#         "conductivity_W_mK": 0.14,
#         "density_kg_m3": 1200,
#         "specific_heat_J_kgK": 650,
#     },
#     "Cast_concrete": {
#         "conductivity_W_mK": 1.13,
#         "density_kg_m3": 1000,
#         "specific_heat_J_kgK": 2000,
#     },
#     "Carpet": {
#         "conductivity_W_mK": 0.06,
#         "density_kg_m3": 1300,
#         "specific_heat_J_kgK": 200,
#     },
#     "Plasterboard": {
#         "conductivity_W_mK": 0.25,
#         "density_kg_m3": 896,
#         "specific_heat_J_kgK": 2800,
#     },
#     "Painted_oak": {
#         "conductivity_W_mK": 0.19,
#         "density_kg_m3": 2390,
#         "specific_heat_J_kgK": 700,
#     },
# }




# beizaee_constructions = {
#     # External cavity wall
#     "external_wall_layer_materials": [
#         "beizaee_brick_outer_leaf",
#         "AIR-GAP",                # you already have an Air Gap material
#         "beizaee_brick_inner_leaf",
#         "beizaee_plaster_dense",
#     ],
#     "external_wall_layer_thickness": [
#         0.105,
#         0.07,
#         0.105,
#         0.013,
#     ],

#     # Internal partition walls
#     "partition_layer_materials": [
#         "beizaee_plaster_dense",
#         "beizaee_brick_inner_leaf",
#         "beizaee_plaster_dense",
#     ],
#     "partition_layer_thickness": [
#         0.013,
#         0.105,
#         0.013,
#     ],

#     # Party wall
#     "party_wall_layer_materials": [
#         "beizaee_plaster_dense",
#         "beizaee_brick_inner_leaf",
#         "AIR-GAP",
#         "beizaee_brick_inner_leaf",
#         "beizaee_plaster_dense",
#     ],
#     "party_wall_layer_thickness": [
#         0.013,
#         0.105,
#         0.07,
#         0.105,
#         0.013,
#     ],

#     # Ground floor (semi-exposed)
#     "ground_floor_layer_materials": [
#         "beizaee_timber_flooring",
#         "beizaee_carpet",
#     ],
#     "ground_floor_layer_thickness": [
#         0.02,
#         0.005,
#     ],

#     # Kitchen solid floor
#     "kitchen_floor_layer_materials": [
#         "beizaee_cast_concrete",
#     ],
#     "kitchen_floor_layer_thickness": [
#         0.1,
#     ],

#     # Internal floor (between GF and FF)
#     "upper_floor_layer_materials": [
#         "beizaee_plasterboard",
#         "AIR-GAP",
#         "beizaee_timber_flooring",
#         "beizaee_carpet",
#     ],
#     "upper_floor_layer_thickness": [
#         0.013,
#         0.1,
#         0.02,
#         0.005,
#     ],

#     # First floor ceiling (semi-exposed)
#     "ceiling_layer_materials": [
#         "beizaee_plasterboard",
#     ],
#     "ceiling_layer_thickness": [
#         0.013,
#     ],

#     # Pitched roof
#     "roof_layer_materials": [
#         "beizaee_clay_tile",
#         "AIR-GAP",
#         "beizaee_roofing_felt",
#     ],
#     "roof_layer_thickness": [
#         0.025,
#         0.02,
#         0.005,
#     ],

#     # Glazing
#     "window_layer_materials": [
#         "beizaee_glazing",
#     ],
#     "window_layer_thickness": [
#         0.003,
#     ],

#     # Window frame
#     "window_frame_layer_materials": [
#         "beizaee_painted_oak",
#     ],
#     "window_frame_layer_thickness": [
#         0.02,
#     ],

#     # Window covered with insulation board
#     "insulated_window_layer_materials": [
#         "beizaee_glazing",
#         "AIR-GAP",
#         "beizaee_polyisocyanurate",
#     ],
#     "insulated_window_layer_thickness": [
#         0.003,
#         0.01,
#         0.05,
#     ],

#     # Doors (internal & external)
#     "door_layer_materials": [
#         "beizaee_painted_oak",
#     ],
#     "door_layer_thickness": [
#         0.044,
#     ],
# }




In [24]:
# import requests

# year = 2013
# region = "Loughborough, UK"
# epw_file_name = "loughborough_beizaee_oiko_2013.epw"
# OIKOLAB_API_KEY = "6f8a9cf2311245a783c97c7275de97b2"

# lat = 52.8
# lon = -1.23

# r = requests.get('https://api.oikolab.com/epw',
#                  params={'lat': lat,
#                          'lon': lon,
#                          'year': year},
#                          headers={'api-key': OIKOLAB_API_KEY}
#                  )



# if r.status_code == 200:
#     with open(epw_file_name, "wb") as f:
#         f.write(r.content)
# else:
#     print(f"Request failed: {r.status_code}, {r.text}")
